# 上市＋上櫃：完整指標大表


- CSV為唯一股票來源，依先前決定暫時排除3718，合計149檔。
- 個股使用FinLab還原價格、成交股數及成交金額；TWSE／TPEx含息市場基準使用FinMind。
- 輸出最後大表及CSV；缺漏、異常與計算留空原因寫入TXT，不補值。
- 後續篩選直接讀取大表。


In [ ]:
%pip -q install finlab pandas requests


## 1. 設定與錯誤紀錄
修改資料期間(預設抓100天的資料)與CSV位置。每天使用一個輸出資料夾，同一天重跑會更新同名檔案；`record()`寫入TXT，`step()`在發生錯誤時先留紀錄再停止。

輸出位置：我的雲端硬碟／玉山競賽2026／股票池／output／日期_stock_indicators。請先完成Google Drive掛載授權；這是固定路徑，未來若移動資料夾，需同步修改output_root。


In [ ]:
# 本格建立紀錄工具與設定；後續各步驟沿用同一個輸出資料夾。
from pathlib import Path
from getpass import getpass
from io import StringIO
from contextlib import contextmanager
import json
import numpy as np
import pandas as pd
import requests
from IPython.display import display, FileLink
# 資料上限預設為台北時間今日；可改成固定日期。
END_DATE = pd.Timestamp.now(tz="Asia/Taipei").date().isoformat()
START_DATE = (pd.Timestamp(END_DATE) - pd.Timedelta(days=100)).date().isoformat()
CSV_PATH = "/content/all_150_stock.csv"

# 掛載Google Drive；首次執行請選擇存放此notebook的帳號並授權。
from google.colab import drive
drive.mount("/content/drive")

# 所有輸出統一放在「玉山競賽2026／股票池／output」。
# 子資料夾只使用執行日期，例如20261103_stock_indicators。
# 同一天重跑沿用資料夾並更新同名檔案；TXT在設定區重跑時重新建立。
output_root = Path("/content/drive/MyDrive/玉山競賽2026/股票池/output")
stamp = pd.Timestamp.now(tz="Asia/Taipei").strftime("%Y%m%d")
out = output_root / f"{stamp}_stock_indicators"
out.mkdir(parents=True, exist_ok=True)
log_file = out / "資料檢查紀錄.txt"
# 每次從設定區重跑先清空舊TXT，後續只追加本次紀錄。
log_file.write_text("", encoding="utf-8")
secrets = []  # 僅用於遮蔽錯誤訊息中的金鑰，不寫入檔案。

def record(message):
    text = str(message)
    for secret in secrets:
        if secret:
            text = text.replace(secret, "[金鑰已隱藏]")
    with log_file.open("a", encoding="utf-8") as f:
        f.write(text + "\n")

@contextmanager
def step(name):
    # 保留錯誤種類與原因，畫面只提示紀錄路徑，避免大量追蹤訊息。
    try:
        yield
    except Exception as exc:
        record(f"錯誤｜{name}｜{type(exc).__name__}：{exc}")
        raise RuntimeError(f"{name}失敗，請查看 {log_file.resolve()}") from None

# 在Colab中可從左側檔案區下載TXT；此連結也會在流程開始時提供。
print("檢查紀錄：", log_file.resolve())
display(FileLink(str(log_file)))


Mounted at /content/drive
檢查紀錄： /content/drive/MyDrive/玉山競賽2026/股票池/output/20261110_stock_indicators/資料檢查紀錄.txt


/content/drive/MyDrive/玉山競賽2026/股票池/output/20261110_stock_indicators/資料檢查紀錄.txt

## 2. 從CSV讀取股票範圍
驗證股票代碼、名稱、市場與重複資料，再暫時排除3718。CSV是唯一名單來源，不在程式中另建名單。


In [ ]:
# 先驗證完整150檔原始名單，再排除指定股票，避免誤用其他CSV。
with step("讀取股票CSV"):
    csv_file = Path(CSV_PATH)
    if not csv_file.is_file():
        from google.colab import files
        uploaded = files.upload()
        names = [name for name in uploaded if name.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError("請只上傳一個股票清單CSV")
        csv_file = Path(names[0])
        csv_file.write_bytes(uploaded[names[0]])
        del uploaded

    # 原檔為Big5／CP950，同時支援UTF-8。
    raw = csv_file.read_bytes()
    for encoding in ["utf-8-sig", "cp950", "big5"]:
        try:
            text = raw.decode(encoding)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError("CSV編碼無法辨識")
    stocks = pd.read_csv(StringIO(text), dtype=str, keep_default_na=False)
    stocks.columns = stocks.columns.str.strip()
    if not {"stock_id", "stock", "market"}.issubset(stocks.columns):
        raise ValueError("CSV需要 stock_id、stock、market 三欄")
    stocks = stocks[["stock_id", "stock", "market"]].apply(lambda x: x.str.strip())
    if stocks.eq("").any().any() or not stocks.stock_id.str.fullmatch(r"\d{4}").all():
        raise ValueError("CSV有空白資料或非4位數股票代碼")
    if stocks.stock_id.duplicated().any():
        raise ValueError("重複股票代碼：" + ",".join(stocks.loc[stocks.stock_id.duplicated(), "stock_id"]))
    stocks["market"] = stocks.market.str.upper().map({"TWSE":"TWSE", "TPEX":"TPEx"})
    if stocks.market.isna().any() or stocks.market.value_counts().to_dict() != {"TWSE":100,"TPEx":50}:
        raise ValueError("請提供上市100檔、上櫃50檔的完整CSV")
    stocks = stocks.loc[stocks.stock_id.ne("3718")].copy()
    record("排除｜3718｜歷史資料不足，依使用者指定暫時排除")
    stock_ids = stocks.stock_id.tolist()


Saving all_150_stock.csv to all_150_stock.csv


## 3. 取得FinLab個股資料
取得還原價格、成交股數（股）、成交金額（元）。還原價格用於計算價格報酬，成交股數用於量能比較；成交金額用於計算最新日及20日日均交易規模。


In [ ]:
# load_stock_data只負責下載原始資料，不檢查完整性或計算指標。
import finlab
from finlab import data

# 不顯示中間資料表；保留還原價格、成交股數與成交金額。
with step("下載FinLab個股資料"):
    key = getpass("FinLab API Key：").strip()
    secrets.append(key)
    if not key:
        raise ValueError("API Key不可空白")
    finlab.login(key)

    def load_stock_data(dataset):
        # 這裡只取得原始資料；日期、數值格式及完整性統一在第5節處理。
        return pd.DataFrame(data.get(dataset)).copy()

    adj_close = load_stock_data("etl:adj_close")
    volume = load_stock_data("price:成交股數")
    amount = load_stock_data("price:成交金額")
    del key


FinLab API Key：··········


/tmp/ipykernel_14229/3634819201.py:11: DeprecationWarning: Passing api_token to finlab.login() is deprecated and will be removed after 2026/08/01. Use finlab.login() (without arguments) to authenticate interactively.
  finlab.login(key)


輸入成功! / Token accepted!
Daily usage: 33.4 / 5000 MB - etl:adj_close


## 4. 取得FinMind市場基準
同一資料集 `TaiwanStockTotalReturnIndex`：`TAIEX`為上市、`TPEx`為櫃買。API的`price`是含息指數點位，之後與個股使用相同起訖日計算報酬。


In [ ]:
# load_benchmark先驗證API結果，再整理成「日期→指數點位」序列。
with step("下載FinMind市場基準"):
    token = getpass("FinMind Token：").strip()
    secrets.append(token)
    if not token:
        raise ValueError("FinMind Token不可空白")

    def load_benchmark(index_id):
        response = requests.get("https://api.finmindtrade.com/api/v4/data",
            headers={"Authorization": f"Bearer {token}"},
            params={"dataset":"TaiwanStockTotalReturnIndex", "data_id":index_id,
                    "start_date":START_DATE, "end_date":END_DATE}, timeout=30)
        response.raise_for_status()
        payload = response.json()
        if payload.get("status") != 200:
            raise ValueError(f"{index_id} API狀態：{payload.get('status')}，請檢查權限或額度")
        frame = pd.DataFrame(payload.get("data", []))
        if frame.empty or not {"date","price","stock_id"}.issubset(frame.columns):
            raise ValueError(index_id + "資料為空或欄位不符")
        # API狀態、欄位及回傳代碼屬下載結果驗證；資料格式及完整性在第5節處理。
        if not frame.stock_id.eq(index_id).all():
            raise ValueError(index_id + "回傳了其他指數代碼")
        return frame.set_index("date")["price"]

    # price欄位為含息報酬指數點位
    benchmarks = pd.concat({"TWSE":load_benchmark("TAIEX"),
                            "TPEx":load_benchmark("TPEx")}, axis=1)
    del token


FinMind Token：··········


from FinMind.data import DataLoader

with step("下載FinMind上櫃市場基準"):
    token = getpass("FinMind Token：").strip()
    secrets.append(token)

    api = DataLoader()
    api.login_by_token(api_token=token)

    benchmark = api.taiwan_stock_total_return_index(
        index_id="TPEx",
        start_date=START_DATE,
        end_date=END_DATE
    ).set_index("date")["price"]

    del token

## 5. 確認資料完整與格式轉換

所有計算所需的檢查都集中在本節：日期、代碼、數值格式、資料長度、缺值與異常值。

1. 統一日期格式、股票代碼字串與數值格式；使用各來源日期聯集對齊。
2. 整體不足61個交易日、重複日期／代碼或市場不明時停止。
3. 缺值／非正價格／非正成交量／負成交金額寫入TXT。
4. 每段期間各自檢查；該段有一天無效，就將該股票「該段計算用資料」留空，不影響其他有效期間。
5. 產生`prepared_data`，供第6節起的計算函式使用。原始資料保留供第13節獨立驗證。

`prepared_data`包含基本資料、兩種成交金額資料、五段個股價格、五段市場指數與三段成交量。後續函式不再檢查、不寫錯誤紀錄。
日期聯集不是官方交易日曆，全部來源同時漏掉的交易日仍無法辨識。零金額可計算，零成交量依目前約定視為無效。

補充：每日資料統一為台北日期，若同一天有多筆資料會停止，不自行平均。格式轉換失敗另記錄原因。
`prepared_data["sources"]`保存轉換格式及對齊後、尚未遮蔽的原始數值，只供驗證使用。
來源缺某一股票或市場欄位時會補出NaN並記錄，受影響指標留空；不會自動用其他股票或市場替代。


In [ ]:
# 將連續的缺漏交易日合併，例：2300 2026/09/22～2026/09/30 無價格資料。
def record_dates(mask, label):
    mask = mask.fillna(False).astype(bool)
    groups = mask.ne(mask.shift()).cumsum()
    for _, group in mask.groupby(groups):
        if group.iloc[0]:
            first, last = group.index[0], group.index[-1]
            period = first.strftime("%Y/%m/%d")
            if first != last:
                period += "～" + last.strftime("%Y/%m/%d")
            record(f"{label}｜{period}")

def prepare_data(stock_list, prices, volumes, amounts, benchmarks):
    """只轉換、對齊、檢查及準備計算資料；不計算指標。"""
    required_columns = {"stock_id", "stock", "market"}
    if not required_columns.issubset(stock_list.columns):
        raise ValueError("股票清單需要stock_id、stock、market三欄")
    stock_info = stock_list.copy()
    if stock_info[["stock_id", "stock", "market"]].isna().any().any():
        raise ValueError("股票清單有空白資料")
    stock_info["stock_id"] = stock_info["stock_id"].astype(str).str.strip()
    if not stock_info["stock_id"].str.fullmatch(r"\d{4}").all():
        raise ValueError("股票代碼必須是4位數字")
    if stock_info.empty or stock_info["stock_id"].duplicated().any():
        raise ValueError("股票清單為空或代碼重複")
    if not stock_info["market"].isin(["TWSE", "TPEx"]).all():
        raise ValueError("股票清單含無法辨識的市場")
    stock_info = stock_info.set_index("stock_id")

    def normalize_frame(frame, source_name):
        # 任何無法轉為數字的內容會成為NaN，之後記錄缺漏日期。
        frame = frame.copy()
        frame.index = pd.to_datetime(frame.index, errors="coerce")
        # 來源都是日資料；帶時區時先轉台北時間，再移除時間部分。
        if frame.index.tz is not None:
            frame.index = frame.index.tz_convert("Asia/Taipei").tz_localize(None)
        frame.index = frame.index.normalize()
        frame.columns = frame.columns.astype(str).str.strip()
        if frame.empty or frame.index.isna().any():
            raise ValueError(source_name + "為空或有無效日期")
        if frame.index.has_duplicates or frame.columns.has_duplicates:
            raise ValueError(source_name + "有重複日期或欄位")
        frame = frame.sort_index().loc[START_DATE:END_DATE]
        if frame.empty:
            raise ValueError(source_name + "在指定期間沒有資料")
        numeric_frame = frame.apply(pd.to_numeric, errors="coerce")
        # 區分原本缺值與無法轉成數字的文字，方便追查來源。
        for code in frame.columns:
            conversion_failed = frame[code].notna() & numeric_frame[code].isna()
            record_dates(conversion_failed, f"格式錯誤｜{code} {source_name}無法轉為數值")
        return numeric_frame.astype(float)

    prices = normalize_frame(prices, "價格")
    volumes = normalize_frame(volumes, "成交量")
    amounts = normalize_frame(amounts, "成交金額")
    benchmarks = normalize_frame(benchmarks, "市場基準")
    dates = prices.index.union(volumes.index).union(amounts.index).union(benchmarks.index).sort_values()
    if len(dates) < 61:
        raise ValueError("整體交易日不足61日，請延長下載期間")
    # 4天是日曆天新鮮度檢查；遇長假需先確認休市情形。
    if (pd.Timestamp(END_DATE) - dates[-1]).days > 4:
        raise ValueError("資料落後要求日期超過4天，請確認更新、權限或長假")
    for name, frame in {"價格": prices, "成交量": volumes, "成交金額": amounts, "市場基準": benchmarks}.items():
        record(f"來源最新日期｜{name}｜{frame.index.max():%Y/%m/%d}")

    prices = prices.reindex(index=dates, columns=stock_info.index)
    volumes = volumes.reindex(index=dates, columns=stock_info.index)
    amounts = amounts.reindex(index=dates, columns=stock_info.index)
    benchmarks = benchmarks.reindex(index=dates, columns=["TWSE", "TPEx"])
    # 保存只轉格式、對齊過的原始數值，供第13節獨立驗證。
    # 後面where產生新表，不會修改這些來源資料。
    source_data = {"prices": prices, "volumes": volumes,
                   "amounts": amounts, "benchmarks": benchmarks}

    def remove_invalid(frame, name, allow_zero=False):
        # 檢查完整下載期間，保留每檔缺漏的實際日期範圍。
        valid = np.isfinite(frame) & (frame.ge(0) if allow_zero else frame.gt(0))
        for code in frame.columns:
            values = frame[code]
            record_dates(values.isna(), f"缺值｜{code} 無{name}資料")
            record_dates(values.notna() & ~valid[code], f"異常｜{code} {name}數值無效")
        return frame.where(valid)

    prices = remove_invalid(prices, "價格")
    volumes = remove_invalid(volumes, "成交量")
    amounts = remove_invalid(amounts, "成交金額", allow_zero=True)
    benchmarks = remove_invalid(benchmarks, "市場指數")

    def prepare_window(window, label):
        # 只遮蔽該段有缺漏的股票，原始資料及其他期間不受影響。
        valid = window.notna().all()
        for code in window.columns[~valid]:
            record(f"計算留空｜{code}｜{label}資料不完整")
        window = window.copy()
        window.loc[:, ~valid] = np.nan
        return window

    def prepare_return_windows(frame, source_name):
        return {
            "近5日": prepare_window(frame.tail(6), source_name + "近5日"),
            "近10日": prepare_window(frame.tail(11), source_name + "近10日"),
            "近20日": prepare_window(frame.tail(21), source_name + "近20日"),
            "近60日": prepare_window(frame.tail(61), source_name + "近60日"),
            "前一段5日": prepare_window(frame.tail(11).head(6), source_name + "前一段5日"),
        }

    return {
        "stock_info": stock_info,
        "dates": dates,
        "sources": source_data,
        "date": dates[-1].strftime("%Y-%m-%d"),
        "amount": {
            "latest": amounts.tail(1),
            "last_20_days": prepare_window(amounts.tail(20), "近20日成交金額"),
        },
        "price": prepare_return_windows(prices, "個股價格"),
        "market": prepare_return_windows(benchmarks, "市場指數"),
        "volume": {
            "latest": volumes.tail(1),
            "last_5_days": prepare_window(volumes.tail(5), "近5日成交量"),
            "previous_20_days": prepare_window(volumes.tail(25).head(20), "此前20日成交量"),
        },
    }


with step("確認資料完整與格式轉換"):
    prepared_data = prepare_data(stocks, adj_close, volume, amount, benchmarks)


# **計算指標**

## 6. 成交金額：觀察交易規模

- 最新日成交金額：最後交易日的成交總金額，單位元。
- 近20日日均成交金額：最近20個交易日成交金額總和÷20，供之後流動性排序使用。
- 明確為0的金額可納入平均；缺值、負值及無限值不可。這裡只計算，不設定入選門檻。

大表與後續篩選統一以千萬元顯示成交金額：原始元÷10,000,000；此換算不影響排名。


In [ ]:
def calculate_amount(amount_data):
    """使用第5節檢查好的金額資料；只計算最新日及20日平均。"""
    latest_amount = amount_data["latest"].iloc[-1]  # 最新日金額，元
    amount_last_20_days = amount_data["last_20_days"]
    average_amount = amount_last_20_days.mean(skipna=False)  # 20日日均金額，元
    result = pd.DataFrame(index=amount_last_20_days.columns)
    # 原始成交金額為元，除以10,000,000換算為千萬元；例如1億元顯示10。
    result["最新日成交金額（千萬元）"] = latest_amount / 10_000_000
    result["近20日日均成交金額（千萬元）"] = average_amount / 10_000_000
    return result


## 7. 個股報酬率：觀察不同期間的價格表現

公式：`（期末還原價格÷期初還原價格－1）×100`，單位%。

| 指標 | 起訖日 | 用意 |
|---|---|---|
| 近5日報酬率 | 5個交易日前→截止日 | 觀察近期方向 |
| 近10日報酬率 | 10個交易日前→截止日 | 觀察短期趨勢 |
| 近20日報酬率 | 20個交易日前→截止日 | 延續原本約一個月的比較期間 |
| 近60日報酬率 | 60個交易日前→截止日 | 觀察較長期間的表現 |
| 前一段5日報酬率 | 10個交易日前→5個交易日前 | 與最近5日比較是否改善，兩段不重疊 |

上述條件統一在第5節確認；第7節只計算報酬。60日資料缺漏不會讓5日報酬一起留空。


In [ ]:
def calculate_one_return(period_prices):
    """輸入已檢查的一段價格，回傳該期間報酬率（%）。"""
    start_price = period_prices.iloc[0]
    end_price = period_prices.iloc[-1]
    # return_percent單位為%；無效資料已在第5節留空，相除仍會留空。
    return_percent = (end_price / start_price - 1) * 100
    return return_percent


def calculate_price_returns(price_data):
    """使用第5節準備好的五段價格，各自計算報酬。"""
    result = pd.DataFrame(index=price_data["近5日"].columns)
    result["近5日報酬率（%）"] = calculate_one_return(price_data["近5日"])
    result["近10日報酬率（%）"] = calculate_one_return(price_data["近10日"])
    result["近20日報酬率（%）"] = calculate_one_return(price_data["近20日"])
    result["近60日報酬率（%）"] = calculate_one_return(price_data["近60日"])
    result["前一段5日報酬率（%）"] = calculate_one_return(price_data["前一段5日"])
    return result


## 8. 市場報酬率：建立相同期間的比較基準

與個股使用完全相同交易日起訖及報酬公式。FinMind提供的是含息指數點位，必須先算漲跌百分比。
上市股票使用TWSE，上櫃使用TPEx；每個市場的各期間報酬只計算一次，再對應至每檔股票。


In [ ]:
def calculate_market_returns(market_data):
    """使用第5節準備好的TWSE／TPEx指數；本段直接呈現報酬公式。"""
    def calculate_return(index_prices):
        start_index = index_prices.iloc[0]  # 期初含息指數點位
        end_index = index_prices.iloc[-1]   # 期末含息指數點位
        return_percent = (end_index / start_index - 1) * 100
        return return_percent

    result = pd.DataFrame(index=market_data["近5日"].columns)
    result["近5日市場報酬率（%）"] = calculate_return(market_data["近5日"])
    result["近10日市場報酬率（%）"] = calculate_return(market_data["近10日"])
    result["近20日市場報酬率（%）"] = calculate_return(market_data["近20日"])
    result["近60日市場報酬率（%）"] = calculate_return(market_data["近60日"])
    result["前一段5日市場報酬率（%）"] = calculate_return(market_data["前一段5日"])
    return result


## 9. 相對市場報酬：判斷是否優於所屬市場

公式：`個股期間報酬率－所屬市場同期報酬率`，單位為百分點。
例如個股跌1%、市場跌3%，相對報酬為＋2個百分點，代表抗跌，不代表個股上漲。
個股或市場報酬有缺值時，對應的相對報酬留空。


In [ ]:
def calculate_relative_returns(stock_returns, market_returns):
    """輸入兩張以股票代碼為索引的表；回傳個股減市場的差距。"""
    # market_returns必須事先對應所屬市場：上市用TWSE、上櫃用TPEx。
    # 每一行都寫出相減欄位；單位為百分點，不是百分比。
    result = pd.DataFrame(index=stock_returns.index)

    # 近5日：個股報酬率－同期間市場報酬率。
    result["近5日相對市場報酬（%）"] = (
        stock_returns["近5日報酬率（%）"]
        - market_returns["近5日市場報酬率（%）"]
    )

    # 近10日：依顯示需求欄名用（%），數值仍為兩個報酬率的百分點差，不另乘除100。
    result["近10日相對市場報酬（%）"] = (
        stock_returns["近10日報酬率（%）"]
        - market_returns["近10日市場報酬率（%）"]
    )

    # 近20日：個股報酬率－同期間市場報酬率。
    result["近20日相對市場報酬（%）"] = (
        stock_returns["近20日報酬率（%）"]
        - market_returns["近20日市場報酬率（%）"]
    )

    # 近60日：個股報酬率－同期間市場報酬率。
    result["近60日相對市場報酬（%）"] = (
        stock_returns["近60日報酬率（%）"]
        - market_returns["近60日市場報酬率（%）"]
    )

    # 前一段5日：個股報酬率－同期間市場報酬率。
    result["前一段5日相對市場報酬（%）"] = (
        stock_returns["前一段5日報酬率（%）"]
        - market_returns["前一段5日市場報酬率（%）"]
    )
    # 任一邊缺值時，相減結果會自然留空，不補成0。
    return result


## 10. 成交量與量能放大比：觀察交易是否升溫

| 指標 | 計算方式 | 用意 |
|---|---|---|
| 最新日成交量（股） | 截止日成交股數 | 觀察當日交易量 |
| 近5日日均成交量（股） | 最近5個交易日平均量 | 觀察近期活躍程度 |
| 此前20日日均成交量（股） | 最近5日之前的20個交易日平均量 | 建立不重疊的比較基期 |
| 量能放大比（倍） | 近5日日均量÷此前20日日均量 | 1.5倍代表增加50%，不代表必然上漲 |

零量、缺量或無限值視為該段無效；兩段各自檢查，放大比需兩段都有效。


In [ ]:
def calculate_volume(volume_data):
    """只計算均量與放大比；各段有效性已由第5節確認。"""
    latest_volume = volume_data["latest"].iloc[-1]
    recent_average_volume = volume_data["last_5_days"].mean(skipna=False)
    baseline_average_volume = volume_data["previous_20_days"].mean(skipna=False)
    # volume_ratio：近5日日均量÷此前20日日均量，單位倍。
    volume_ratio = recent_average_volume / baseline_average_volume
    result = pd.DataFrame(index=volume_data["last_5_days"].columns)
    result["最新日成交量（股）"] = latest_volume
    result["近5日日均成交量（股）"] = recent_average_volume
    result["此前20日日均成交量（股）"] = baseline_average_volume
    result["量能放大比（倍）"] = volume_ratio
    return result


## 11. 合成上市＋上櫃指標大表

`calculate_all_indicators()`負責呼叫前面的計算函式，依股票代碼合併結果。
每檔股票一列，順序為股票代碼、股票名稱、資料截止日、市場、成交金額、各期間個股／市場／相對報酬，以及成交量指標。
所有股票使用同一截止日。歷史資料不足或部分缺值保留該股票，只留空受影響的指標。


In [ ]:
def calculate_all_indicators(prepared_data):
    """只接收第5節的結果，呼叫計算函式並合成大表。"""
    stock_info = prepared_data["stock_info"]
    amount_result = calculate_amount(prepared_data["amount"])
    stock_returns = calculate_price_returns(prepared_data["price"])
    market_summary = calculate_market_returns(prepared_data["market"])
    volume_result = calculate_volume(prepared_data["volume"])

    # 3. 將TWSE／TPEx市場報酬分配給所屬股票。
    # 例如2330的市場為TWSE，就取市場報酬總表中的TWSE那一列。
    stock_market_returns = pd.DataFrame(index=stock_info.index)
    for column in market_summary.columns:
        stock_market_returns[column] = stock_info["market"].map(market_summary[column])

    # 4. 此時兩張表都以股票代碼為索引，可以逐檔計算差距。
    relative_returns = calculate_relative_returns(stock_returns, stock_market_returns)

    # 5. 建立基本資料，再按股票代碼合併各項結果。
    result = stock_info[["stock", "market"]].rename(
        columns={"stock": "股票名稱", "market": "市場"})
    result["資料截止日"] = prepared_data["date"]
    result = result.join(amount_result)
    result = result.join(stock_returns)
    result = result.join(stock_market_returns)
    result = result.join(relative_returns)
    result = result.join(volume_result)

    # 6. 明確列出閱讀順序：每個期間的三種報酬放在一起。
    column_order = [
        "股票名稱",
        "資料截止日",
        "市場",
        "最新日成交金額（千萬元）",
        "近20日日均成交金額（千萬元）",
        "近5日報酬率（%）",
        "近5日市場報酬率（%）",
        "近5日相對市場報酬（%）",
        "近10日報酬率（%）",
        "近10日市場報酬率（%）",
        "近10日相對市場報酬（%）",
        "近20日報酬率（%）",
        "近20日市場報酬率（%）",
        "近20日相對市場報酬（%）",
        "近60日報酬率（%）",
        "近60日市場報酬率（%）",
        "近60日相對市場報酬（%）",
        "前一段5日報酬率（%）",
        "前一段5日市場報酬率（%）",
        "前一段5日相對市場報酬（%）",
        "最新日成交量（股）",
        "近5日日均成交量（股）",
        "此前20日日均成交量（股）",
        "量能放大比（倍）",
    ]
    result = result[column_order]
    return result.rename_axis("股票代碼").reset_index()


## 12. 執行與輸出：只顯示最後大表

執行後得到`final_result`，包含上市及上櫃全部股票，後續篩選可直接使用。
CSV保留計算精度，畫面只顯示到小數第3位；TXT記錄資料缺漏及錯誤。
請從設定區重新執行，確保已下載足夠的180個日曆天資料。

輸出在Colab執行環境內，環境重設後可能消失，請下載CSV與TXT。資料截止日是共同計算日期，不保證每檔股票該日都有有效資料；缺值會留空。


In [ ]:
with step("上市與上櫃指標計算及輸出"):
    final_result = calculate_all_indicators(prepared_data)
    result_file = out / "上市上櫃完整指標大表.csv"
    final_result.to_csv(result_file, index=False, encoding="utf-8-sig")
    record(f"完成｜上市＋上櫃指標大表｜{len(final_result)}檔｜截止日{final_result['資料截止日'].iloc[0]}")
    display(final_result.round(3))
    display(FileLink(str(result_file)))
    display(FileLink(str(log_file)))


,股票代碼,股票名稱,資料截止日,市場,最新日成交金額（千萬元）,近20日日均成交金額（千萬元）,近5日報酬率（%）,近5日市場報酬率（%）,近5日相對市場報酬（%）,近10日報酬率（%）,...,近60日報酬率（%）,近60日市場報酬率（%）,近60日相對市場報酬（%）,前一段5日報酬率（%）,前一段5日市場報酬率（%）,前一段5日相對市場報酬（%）,最新日成交量（股）,近5日日均成交量（股）,此前20日日均成交量（股）,量能放大比（倍）
0,2330,台積電,2026-11-10,TWSE,8249.185,9277.088,-0.436,-1.375,0.940,-4.792,...,-4.392,-5.108,0.716,-4.375,-6.049,1.674,36418797.0,58414223.6,30341454.25,1.925
1,2454,聯發科,2026-11-10,TWSE,7631.802,4575.223,4.218,-1.375,5.593,-3.283,...,-4.726,-5.108,0.382,-7.197,-6.049,-1.148,19554575.0,16606831.6,9377696.75,1.771
2,2308,台達電,2026-11-10,TWSE,2071.797,2323.804,0.394,-1.375,1.769,-15.563,...,-29.363,-5.108,-24.255,-15.894,-6.049,-9.845,16283594.0,22866859.0,13500789.35,1.694
3,2317,鴻海,2026-11-10,TWSE,1101.575,1105.388,-1.629,-1.375,-0.254,-2.424,...,-8.175,-5.108,-3.067,-0.808,-6.049,5.241,45344218.0,59662102.0,43695832.15,1.365
4,3711,日月光投控,2026-11-10,TWSE,2042.928,1528.932,-1.786,-1.375,-0.410,-7.066,...,-3.816,-5.108,1.292,-5.376,-6.049,0.672,33676365.0,34257456.6,18484012.55,1.853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,4772,台特化,2026-11-10,TPEx,22.280,16.760,1.636,-3.630,5.266,-11.943,...,-13.519,-14.777,1.258,-13.360,-8.881,-4.480,1022987.0,829041.6,1047671.55,0.791
145,8932,智通,2026-11-10,TPEx,10.882,14.953,-5.674,-3.630,-2.044,-8.904,...,-8.800,-14.777,5.977,-3.425,-8.881,5.456,1352838.0,1862448.0,1532415.00,1.215
146,6584,南俊國際,2026-11-10,TPEx,43.299,18.280,-16.133,-3.630,-12.503,-26.690,...,-41.353,-14.777,-26.577,-12.587,-8.881,-3.707,1394088.0,959577.2,386081.80,2.485
147,3227,原相,2026-11-10,TPEx,14.569,16.331,1.714,-3.630,5.344,-3.784,...,-14.833,-14.777,-0.056,-5.405,-8.881,3.475,817323.0,1075421.4,818617.15,1.314


/content/drive/MyDrive/玉山競賽2026/股票池/output/20261110_stock_indicators/上市上櫃完整指標大表.csv

/content/drive/MyDrive/玉山競賽2026/股票池/output/20261110_stock_indicators/資料檢查紀錄.txt

# 篩選股票池
下方成交金額統一使用「千萬元」，沿用大表已換算的數值，不再除以1,000萬。近10日相對市場報酬欄名使用「（%）」，數值仍是個股與市場報酬率相減的差距。


## 四種候選池方法比較

| 方法 | 使用指標 | 篩選規則 | 標的涵蓋範圍 | 用途 | 選股面向 |
|---|---|---|---|---|---|
| 方法1：成交金額＋20日相對報酬 | 近20日日均成交金額、近20日相對市場報酬 | 先按日均成交金額由大到小取前60檔，再在其中按20日相對報酬由大到小取前40檔 | 上市＋上櫃合併篩選，不分配各市場名額 | 從交易規模較大的股票中，找出近一個月相對表現較佳者 | 交易規模／流動性代理指標＋約一個月的價格動能 |
| 方法2：成交金額＋60日相對報酬 | 近20日日均成交金額、近60日相對市場報酬 | 同樣先取成交金額前60檔，再在其中按60日相對報酬由大到小取前40檔 | 上市＋上櫃合併篩選，不分配各市場名額 | 找出較長期間相對表現較佳者，與方法1比較動能期間差異 | 交易規模／流動性代理指標＋約三個月的價格動能 |
| 方法3：持續強勢 | 近10日、近5日相對市場報酬 | 近10日相對報酬 > 0，且近5日相對報酬 > 0；符合者全部保留 | 上市＋上櫃皆可入選，不限制總檔數及各市場名額 | 保留兩個近期觀察期間都優於所屬市場的股票，不要求放量 | 短期相對強勢；兩段期間重疊，並非逐日皆優於市場 |
| 方法4：近期放量轉強 | 近5日相對市場報酬、前一段5日相對市場報酬、量能放大比 | 近5日相對報酬 > 0，且高於前一段5日相對報酬，並且量能放大比 > 1；符合者全部保留 | 上市＋上櫃皆可入選，不限制總檔數及各市場名額 | 找出近期相對表現改善、且交易比自身基期活躍的股票 | 短期相對動能改善＋成交量活躍度 |

**共同口徑與閱讀重點：**

- 相對市場報酬＝個股同期報酬率－所屬市場同期報酬率；上市對TWSE，上櫃對TPEx，單位為百分點。
- 量能放大比＝近5日日均成交量÷此前20日日均成交量，兩段不重疊；前一段5日報酬則是10個交易日前至5個交易日前。
- 方法1、2是排名選股，不要求相對報酬為正；方法3、4要求相對報酬為正，但個股本身仍可能下跌。
- 各方法只排除自身必要指標缺值的股票。方法1、2不足40檔時不放寬條件、不從成交金額第61名後遞補，原因寫入TXT。
- 成交金額是交易規模的代理指標，不能單獨代表實際可成交容量；本表的用途是建立候選池，尚未決定買進與權重。
- 下方「實驗」另將範圍限於上市股票，重跑方法1、2各取最多40檔，再比較交集與非交集；與上表預設的上市＋上櫃範圍不同。


## 13. 篩選規則與輸入

直接使用第12節的`final_result`，不重新計算指標，也不改動原始大表。
上市、上櫃合併篩選，不分配各市場名額；相對報酬已使用各自所屬市場。

| 方法 | 條件 |
|---|---|
| 方法1 | 近20日日均成交金額前60檔，再取近20日相對市場報酬前40檔 |
| 方法2 | 同一成交金額前60檔，再取近60日相對市場報酬前40檔 |
| 方法3：持續強勢 | 近10日相對報酬 > 0，且近5日相對報酬 > 0 |
| 方法4：近期放量轉強 | 近5日相對報酬 > 0、高於前一段5日相對報酬，且量能放大比 > 1 |

- 方法1、2先決定成交金額前60；其中報酬缺值者不參與第二階段，不從成交金額第61名以後遞補。
- 方法1、2按排名取最多40檔，不額外要求報酬大於0；不足時保留實際檔數、寫入TXT。
- 方法3、4條件必須全部成立，所需指標缺值就不入選；符合的全部保留，不限檔數。
- 同分時：成交金額排序按股票代碼；相對報酬排序按成交金額、股票代碼，確保重跑一致。
- 四種方法各自獨立，一檔股票可同時出現在不同方法中，不自動合併成交易清單。


In [ ]:
# 篩選前只確認大表結構；原始行情的完整性已在第5節處理。
with step("確認篩選輸入"):
    required_columns = [
        "股票代碼", "股票名稱", "資料截止日", "市場",
        "近20日日均成交金額（千萬元）", "近20日相對市場報酬（%）",
        "近60日相對市場報酬（%）", "近10日相對市場報酬（%）",
        "近5日相對市場報酬（%）", "前一段5日相對市場報酬（%）",
        "量能放大比（倍）",
    ]
    if not set(required_columns).issubset(final_result.columns):
        raise ValueError("大表缺少篩選必要欄位，請先執行第12節")
    if final_result.empty or final_result["股票代碼"].duplicated().any():
        raise ValueError("大表為空或股票代碼重複")
    if final_result["資料截止日"].isna().any() or final_result["資料截止日"].nunique() != 1:
        raise ValueError("大表需使用同一個資料截止日")


## 14. 方法1：成交金額前60 → 20日相對報酬前40

`select_amount_relative_20d()`只回傳排名、股票代碼、名稱、日期、市場及本方法使用的兩個指標。
`amount_top_n=60`、`final_n=40`可在呼叫時調整。


In [ ]:
def select_amount_relative_20d(table, amount_top_n=60, final_n=40):
    """先選交易金額較大的股票，再依20日相對市場表現排序。"""
    if not (isinstance(amount_top_n, int) and isinstance(final_n, int)
            and 0 < final_n <= amount_top_n):
        raise ValueError("需符合0 < 最終檔數 <= 成交金額候選檔數")

    # 1. 排除日均成交金額缺值，合併上市、上櫃取前60檔。
    amount_candidates = table.dropna(subset=["近20日日均成交金額（千萬元）"])
    amount_candidates = amount_candidates.sort_values(
        ["近20日日均成交金額（千萬元）", "股票代碼"], ascending=[False, True])
    amount_candidates = amount_candidates.head(amount_top_n)

    # 2. 只在這60檔內排除相對報酬缺值，再取排名前40。
    return_candidates = amount_candidates.dropna(subset=["近20日相對市場報酬（%）"])
    return_candidates = return_candidates.sort_values(
        ["近20日相對市場報酬（%）", "近20日日均成交金額（千萬元）", "股票代碼"],
        ascending=[False, False, True])
    selected = return_candidates.head(final_n).copy()
    selected.insert(0, "排名", range(1, len(selected) + 1))

    # 3. TXT記錄各階段數量與缺值股票，不顯示中間表。
    missing_amount = table.loc[table["近20日日均成交金額（千萬元）"].isna(), "股票代碼"]
    missing_return = amount_candidates.loc[
        amount_candidates["近20日相對市場報酬（%）"].isna(), "股票代碼"]
    for stock_id in missing_amount:
        record(f"篩選排除｜方法1｜{stock_id}｜20日日均成交金額缺值")
    for stock_id in missing_return:
        record(f"篩選排除｜方法1｜{stock_id}｜20日相對市場報酬缺值")
    record(f"方法1｜成交金額候選{len(amount_candidates)}檔｜報酬有效{len(return_candidates)}檔｜入選{len(selected)}檔")
    if len(selected) < final_n:
        record(f"候選不足｜方法1｜實際{len(selected)}檔，不補足至{final_n}檔")
    # 畫面及CSV只保留基本資料與本方法實際使用的指標。
    output_columns = ['排名', '股票代碼', '股票名稱', '資料截止日', '市場', '近20日日均成交金額（千萬元）', '近20日相對市場報酬（%）']
    return selected[output_columns].reset_index(drop=True)


## 15. 方法2：成交金額前60 → 60日相對報酬前40

`select_amount_relative_60d()`只回傳排名、股票代碼、名稱、日期、市場及本方法使用的兩個指標。
`amount_top_n=60`、`final_n=40`可在呼叫時調整。


In [ ]:
def select_amount_relative_60d(table, amount_top_n=60, final_n=40):
    """先選交易金額較大的股票，再依60日相對市場表現排序。"""
    if not (isinstance(amount_top_n, int) and isinstance(final_n, int)
            and 0 < final_n <= amount_top_n):
        raise ValueError("需符合0 < 最終檔數 <= 成交金額候選檔數")

    # 1. 排除日均成交金額缺值，合併上市、上櫃取前60檔。
    amount_candidates = table.dropna(subset=["近20日日均成交金額（千萬元）"])
    amount_candidates = amount_candidates.sort_values(
        ["近20日日均成交金額（千萬元）", "股票代碼"], ascending=[False, True])
    amount_candidates = amount_candidates.head(amount_top_n)

    # 2. 只在這60檔內排除相對報酬缺值，再取排名前40。
    return_candidates = amount_candidates.dropna(subset=["近60日相對市場報酬（%）"])
    return_candidates = return_candidates.sort_values(
        ["近60日相對市場報酬（%）", "近20日日均成交金額（千萬元）", "股票代碼"],
        ascending=[False, False, True])
    selected = return_candidates.head(final_n).copy()
    selected.insert(0, "排名", range(1, len(selected) + 1))

    # 3. TXT記錄各階段數量與缺值股票，不顯示中間表。
    missing_amount = table.loc[table["近20日日均成交金額（千萬元）"].isna(), "股票代碼"]
    missing_return = amount_candidates.loc[
        amount_candidates["近60日相對市場報酬（%）"].isna(), "股票代碼"]
    for stock_id in missing_amount:
        record(f"篩選排除｜方法2｜{stock_id}｜20日日均成交金額缺值")
    for stock_id in missing_return:
        record(f"篩選排除｜方法2｜{stock_id}｜60日相對市場報酬缺值")
    record(f"方法2｜成交金額候選{len(amount_candidates)}檔｜報酬有效{len(return_candidates)}檔｜入選{len(selected)}檔")
    if len(selected) < final_n:
        record(f"候選不足｜方法2｜實際{len(selected)}檔，不補足至{final_n}檔")
    # 畫面及CSV只保留基本資料與本方法實際使用的指標。
    output_columns = ['排名', '股票代碼', '股票名稱', '資料截止日', '市場', '近20日日均成交金額（千萬元）', '近60日相對市場報酬（%）']
    return selected[output_columns].reset_index(drop=True)


## 16. 方法3：持續強勢
近10日與近5日相對市場報酬都必須大於0，不要求放量，不限制檔數。


In [ ]:
def select_continued_strength(table):
    """選出近期兩個期間都優於所屬市場的股票。"""
    required = ["近10日相對市場報酬（%）", "近5日相對市場報酬（%）"]
    candidates = table.dropna(subset=required)
    relative_10_days = candidates["近10日相對市場報酬（%）"]
    relative_5_days = candidates["近5日相對市場報酬（%）"]
    selected = candidates.loc[(relative_10_days > 0) & (relative_5_days > 0)].copy()
    # 排序僅方便閱讀，不限制入選數量。
    selected = selected.sort_values("股票代碼")
    # 此處排名是按股票代碼的顯示序號，不是強弱評分。
    selected.insert(0, "排名", range(1, len(selected) + 1))
    for stock_id in table.loc[table[required].isna().any(axis=1), "股票代碼"]:
        record(f"篩選排除｜方法3｜{stock_id}｜5日或10日相對市場報酬缺值")
    record(f"方法3｜入選{len(selected)}檔")
    # 畫面及CSV只保留基本資料與本方法實際使用的指標。
    output_columns = ['排名', '股票代碼', '股票名稱', '資料截止日', '市場', '近10日相對市場報酬（%）', '近5日相對市場報酬（%）']
    return selected[output_columns].reset_index(drop=True)


## 17. 方法4：近期放量轉強
最近5日優於市場，且相對表現比前一段5日改善，同時量能放大比大於1，不限制檔數。


In [ ]:
def select_volume_turnaround(table):
    """選出近期相對表現改善，且交易量升溫的股票。"""
    required = ["近5日相對市場報酬（%）", "前一段5日相對市場報酬（%）", "量能放大比（倍）"]
    candidates = table.dropna(subset=required)
    relative_5_days = candidates["近5日相對市場報酬（%）"]
    relative_prior_5_days = candidates["前一段5日相對市場報酬（%）"]
    volume_ratio = candidates["量能放大比（倍）"]
    selected = candidates.loc[
        (relative_5_days > 0)
        & (relative_5_days > relative_prior_5_days)
        & (volume_ratio > 1)
    ].copy()
    selected = selected.sort_values("股票代碼")
    # 此處排名是按股票代碼的顯示序號，不是強弱評分。
    selected.insert(0, "排名", range(1, len(selected) + 1))
    for stock_id in table.loc[table[required].isna().any(axis=1), "股票代碼"]:
        record(f"篩選排除｜方法4｜{stock_id}｜5日相對報酬、前段相對報酬或量能比缺值")
    record(f"方法4｜入選{len(selected)}檔")
    # 畫面及CSV只保留基本資料與本方法實際使用的指標。
    output_columns = ['排名', '股票代碼', '股票名稱', '資料截止日', '市場', '近5日相對市場報酬（%）', '前一段5日相對市場報酬（%）', '量能放大比（倍）']
    return selected[output_columns].reset_index(drop=True)


## 18. 執行四種方法與輸出

四個變數`selected_1`～`selected_4`分別保存四種入選名單。
`selection_comparison`保存所有股票的原指標及四個入選標記；False只表示本方法未入選，資料不足原因請查看TXT。
每種方法各存一份CSV，另存入選對照表；原始`final_result`保持不變。

四種方法的畫面與CSV只輸出基本資料及使用指標。方法1、2排名為相對報酬排序；方法3、4排名為按股票代碼排列的顯示序號。


In [ ]:
with step("四種篩選方式及輸出"):
    selected_1 = select_amount_relative_20d(final_result, amount_top_n=60, final_n=40)
    selected_2 = select_amount_relative_60d(final_result, amount_top_n=60, final_n=40)
    selected_3 = select_continued_strength(final_result)
    selected_4 = select_volume_turnaround(final_result)

    # 只建立副本，原始指標大表不加入篩選欄位。
    selection_comparison = final_result.copy()
    selection_comparison["方法1入選"] = final_result["股票代碼"].isin(selected_1["股票代碼"])
    selection_comparison["方法2入選"] = final_result["股票代碼"].isin(selected_2["股票代碼"])
    selection_comparison["方法3入選"] = final_result["股票代碼"].isin(selected_3["股票代碼"])
    selection_comparison["方法4入選"] = final_result["股票代碼"].isin(selected_4["股票代碼"])

    # 只輸出最後名單，不顯示成交金額前60等中間表。
    outputs = [
        ("方法1：成交金額＋20日相對報酬", "method_1_amount_relative_20d.csv", selected_1),
        ("方法2：成交金額＋60日相對報酬", "method_2_amount_relative_60d.csv", selected_2),
        ("方法3：持續強勢", "method_3_continued_strength.csv", selected_3),
        ("方法4：近期放量轉強", "method_4_volume_turnaround.csv", selected_4),
    ]
    for title, filename, selected in outputs:
        result_file = out / filename
        selected.to_csv(result_file, index=False, encoding="utf-8-sig")
        print(f"{title}：{len(selected)}檔")
        display(selected.round(3))
        display(FileLink(str(result_file)))

    comparison_file = out / "selection_comparison.csv"
    selection_comparison.to_csv(comparison_file, index=False, encoding="utf-8-sig")
    print("四種方法入選對照表：")
    display(selection_comparison[["股票代碼", "股票名稱", "資料截止日", "市場",
                                  "方法1入選", "方法2入選", "方法3入選", "方法4入選"]])
    display(FileLink(str(comparison_file)))
    display(FileLink(str(log_file)))


方法1：成交金額＋20日相對報酬：40檔


,排名,股票代碼,股票名稱,資料截止日,市場,近20日日均成交金額（千萬元）,近20日相對市場報酬（%）
0,1,2603,長榮,2026-11-10,TWSE,413.990,23.278
1,2,2891,中信金,2026-11-10,TWSE,340.291,22.896
2,3,1802,台玻,2026-11-10,TWSE,955.773,22.619
3,4,6446,藥華藥,2026-11-10,TWSE,348.567,20.499
4,5,1815,富喬,2026-11-10,TPEx,1606.505,16.770
5,6,2881,富邦金,2026-11-10,TWSE,320.312,13.386
6,7,2317,鴻海,2026-11-10,TWSE,1105.388,10.374
7,8,6415,矽力-KY,2026-11-10,TWSE,342.971,8.629
8,9,3374,精材,2026-11-10,TPEx,569.045,7.368
9,10,1303,南亞,2026-11-10,TWSE,3078.784,6.612


/content/drive/MyDrive/玉山競賽2026/股票池/output/20261110_stock_indicators/method_1_amount_relative_20d.csv

方法2：成交金額＋60日相對報酬：40檔


,排名,股票代碼,股票名稱,資料截止日,市場,近20日日均成交金額（千萬元）,近60日相對市場報酬（%）
0,1,1815,富喬,2026-11-10,TPEx,1606.505,66.975
1,2,1303,南亞,2026-11-10,TWSE,3078.784,57.213
2,3,3443,創意,2026-11-10,TWSE,1760.901,49.810
3,4,3653,健策,2026-11-10,TWSE,1155.056,47.543
4,5,3105,穩懋,2026-11-10,TPEx,1235.243,45.370
5,6,3374,精材,2026-11-10,TPEx,569.045,41.872
6,7,3324,雙鴻,2026-11-10,TPEx,552.133,32.958
7,8,6505,台塑化,2026-11-10,TWSE,482.156,31.128
8,9,1301,台塑,2026-11-10,TWSE,519.216,30.827
9,10,1802,台玻,2026-11-10,TWSE,955.773,29.974


/content/drive/MyDrive/玉山競賽2026/股票池/output/20261110_stock_indicators/method_2_amount_relative_60d.csv

... (6 outputs left out)

四種方法入選對照表：


,股票代碼,股票名稱,資料截止日,市場,方法1入選,方法2入選,方法3入選,方法4入選
0,2330,台積電,2026-11-10,TWSE,True,True,True,False
1,2454,聯發科,2026-11-10,TWSE,True,True,True,True
2,2308,台達電,2026-11-10,TWSE,False,False,False,True
3,2317,鴻海,2026-11-10,TWSE,True,True,False,False
4,3711,日月光投控,2026-11-10,TWSE,True,True,False,False
...,...,...,...,...,...,...,...,...
144,4772,台特化,2026-11-10,TPEx,False,False,True,False
145,8932,智通,2026-11-10,TPEx,False,False,False,False
146,6584,南俊國際,2026-11-10,TPEx,False,False,False,False
147,3227,原相,2026-11-10,TPEx,False,False,True,True


/content/drive/MyDrive/玉山競賽2026/股票池/output/20261110_stock_indicators/selection_comparison.csv

/content/drive/MyDrive/玉山競賽2026/股票池/output/20261110_stock_indicators/資料檢查紀錄.txt

## 19. 各方法上市／上櫃入選檔數

先執行四種篩選，再執行本段。各方法分別統計，不跨方法去重；合計是該方法的入選股票數。


In [ ]:
def summarize_market_counts(selected_1, selected_2, selected_3, selected_4):
    """統計四種方法各自入選的上市、上櫃檔數；不同方法分開計算。"""
    methods = [
        ("方法1：成交金額＋20日相對報酬", selected_1),
        ("方法2：成交金額＋60日相對報酬", selected_2),
        ("方法3：持續強勢", selected_3),
        ("方法4：近期放量轉強", selected_4),
    ]
    rows = []
    for method_name, selected in methods:
        # 市場欄位沿用CSV的TWSE／TPEx；沒有入選股票的市場顯示0。
        listed_count = int(selected["市場"].eq("TWSE").sum())
        otc_count = int(selected["市場"].eq("TPEx").sum())
        rows.append({
            "篩選方法": method_name,
            "上市檔數": listed_count,
            "上櫃檔數": otc_count,
            "合計檔數": len(selected),
        })
    return pd.DataFrame(rows)


with step("各方法上市與上櫃檔數統計"):
    market_counts = summarize_market_counts(selected_1, selected_2, selected_3, selected_4)
    counts_file = out / "selection_market_counts.csv"
    market_counts.to_csv(counts_file, index=False, encoding="utf-8-sig")
    display(market_counts)
    display(FileLink(str(counts_file)))


,篩選方法,上市檔數,上櫃檔數,合計檔數
0,方法1：成交金額＋20日相對報酬,28,12,40
1,方法2：成交金額＋60日相對報酬,30,10,40
2,方法3：持續強勢,20,19,39
3,方法4：近期放量轉強,16,15,31


/content/drive/MyDrive/玉山競賽2026/股票池/output/20261110_stock_indicators/selection_market_counts.csv

## 四種方法兩兩重疊的股票檔數

使用原本上市＋上櫃合併篩選的`selected_1`～`selected_4`，不是實驗1的上市限定名單。
先執行四種篩選，再執行本段；只顯示，不存檔。

- 表格每格＝該列方法與該欄方法都入選的股票數，以股票代碼比對。
- 對角線＝各方法自己的入選檔數；上下對稱的數字相同。
- 例如方法1與方法3交叉處為25，表示兩種方法都選到25檔；不表示這25檔也都被其他方法選到。

左側方法名稱直接附上原本入選檔數，例如「方法1：20日相對報酬（40檔）」；右側各欄顯示兩方法共同入選幾檔。


In [ ]:
def count_method_overlap(selected_1, selected_2, selected_3, selected_4):
    """以股票代碼計算四種方法兩兩交集的檔數，回傳列名稱含入選檔數的兩兩重疊表格。"""
    # set會去重，每檔股票在同一組比較中只計算一次。
    method_stocks = {
        "方法1：20日相對報酬": set(selected_1["股票代碼"].dropna().astype(str).str.strip()),
        "方法2：60日相對報酬": set(selected_2["股票代碼"].dropna().astype(str).str.strip()),
        "方法3：持續強勢": set(selected_3["股票代碼"].dropna().astype(str).str.strip()),
        "方法4：近期放量轉強": set(selected_4["股票代碼"].dropna().astype(str).str.strip()),
    }
    overlap = pd.DataFrame(0, index=method_stocks, columns=method_stocks)
    for row_method, row_stocks in method_stocks.items():
        for column_method, column_stocks in method_stocks.items():
            # &表示取交集；len計算共同股票有幾檔。
            overlap.loc[row_method, column_method] = len(row_stocks & column_stocks)
    # 入選檔數直接放在左側方法名稱後面，右側維持4×4重疊檔數。
    overlap.index = [
        f"{method_name}（{len(stock_ids)}檔）"
        for method_name, stock_ids in method_stocks.items()
    ]
    overlap.index.name = "重疊檔數"
    return overlap



overlap_counts = count_method_overlap(selected_1, selected_2, selected_3, selected_4)
display(overlap_counts)


,方法1：20日相對報酬,方法2：60日相對報酬,方法3：持續強勢,方法4：近期放量轉強
重疊檔數,,,,
方法1：20日相對報酬（40檔）,40,35,12,9
方法2：60日相對報酬（40檔）,35,40,12,8
方法3：持續強勢（39檔）,12,12,39,14
方法4：近期放量轉強（31檔）,9,8,14,31
